You're already achieving **a strong log loss of 0.0473**, which is quite good! However, there are still several ways to **improve your model’s performance**. Here are **some key areas** to focus on:

---

## **1️⃣ Feature Engineering: Extract More Insights from Data**
Adding more meaningful features can make a huge difference. Here are a few ideas:

### 🏀 **Game-Based Features**
- **Average Margin of Victory:** How much a team wins by on average.
- **Win Streaks:** Number of consecutive games won before the tournament.
- **Head-to-Head Performance:** If two teams have played before, how did they perform?
- **Performance Against Top Teams:** Wins against teams with **low seed numbers** (e.g., 1-4).

### 🏀 **Efficiency Metrics**
- **Offensive & Defensive Ratings:** Points per possession.
- **Turnover Rate:** Turnovers per possession.
- **Rebound Efficiency:** Percentage of rebounds grabbed.

### 🏀 **Team Strength Adjustments**
- **Seed Strength Beyond Seeding:** Using ranking systems like **KenPom, Sagarin, or Massey Ordinals** instead of just seed numbers.
- **Adjusted Win Percentage:** Instead of pure wins, adjust for opponent difficulty.

---

## **2️⃣ Model Optimization: Tune Hyperparameters**
Your **best model (XGBoost, log loss: 0.04169)** can be further improved by tuning its hyperparameters.

### 🔥 **Try a Hyperparameter Grid Search**
```python
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier()

param_grid = {
    "n_estimators": [100, 200, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 0.8, 1.0]
}

grid_search = GridSearchCV(xgb, param_grid, cv=5, scoring="neg_log_loss", verbose=1, n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best log loss:", -grid_search.best_score_)
```
✅ This will **find the best settings for XGBoost** and potentially improve log loss.

---

## **3️⃣ Add More Data: Use Regular Season Detailed Stats**
Right now, your model only looks at **compact results**. But the dataset includes **detailed game stats** like:
- **Field Goals Made (FGM)**
- **Turnovers (TO)**
- **Assists (Ast)**
- **Steals (Stl)**
- **Personal Fouls (PF)**

### 🔥 **Integrate `MRegularSeasonDetailedResults.csv`**
```python
mdetailed_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))

# Compute advanced stats per team
def compute_advanced_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WFGM": "mean",  # Field Goals Made
        "WFGM3": "mean",  # 3-Point Field Goals Made
        "WTO": "mean",    # Turnovers
        "WStl": "mean",   # Steals
        "WBlk": "mean",   # Blocks
        "WPF": "mean"     # Personal Fouls
    }).reset_index()

    # Rename
    team_stats.columns = ["Season", "TeamID", "FGM", "FGM3", "Turnovers", "Steals", "Blocks", "Fouls"]
    return team_stats

madvanced_stats = compute_advanced_stats(mdetailed_results)

# Merge with training data
mtrain_data = mtrain_data.merge(madvanced_stats, on=["Season", "WTeamID"], how="left")
mtrain_data = mtrain_data.merge(madvanced_stats, on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

# Create additional features
mtrain_data["FGMDiff"] = mtrain_data["FGM_W"] - mtrain_data["FGM_L"]
mtrain_data["TurnoverDiff"] = mtrain_data["Turnovers_W"] - mtrain_data["Turnovers_L"]
mtrain_data["StealDiff"] = mtrain_data["Steals_W"] - mtrain_data["Steals_L"]
mtrain_data["BlockDiff"] = mtrain_data["Blocks_W"] - mtrain_data["Blocks_L"]

# Update training dataset
X = mtrain_data.drop(columns=["Win"])
y = mtrain_data["Win"]
```
✅ **Adding this should improve model accuracy** by incorporating actual gameplay details.

---

## **4️⃣ Try Neural Networks**
Since you have a strong structured dataset, a deep learning model like **TensorFlow/Keras** could improve accuracy.

```python
from tensorflow import keras
from tensorflow.keras import layers

# Define Neural Network Model
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

# Compile & Train
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_train_scaled, y_train, epochs=20, batch_size=32, validation_data=(X_valid_scaled, y_valid))

# Evaluate
y_pred_nn = model.predict(X_valid_scaled).flatten()
log_loss_nn = log_loss(y_valid, y_pred_nn)
print(f"Neural Network Log Loss: {log_loss_nn}")
```
✅ **Neural networks may capture complex patterns that traditional models miss**.

---

## **5️⃣ Try an Ensemble Model (Stacking)**
Rather than picking the **best** model, why not **combine multiple models**?

### 🔥 **Stacking Classifier**
```python
from sklearn.ensemble import StackingClassifier

stacked_model = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200)),
        ("gb", GradientBoostingClassifier(n_estimators=200)),
        ("xgb", XGBClassifier(n_estimators=200))
    ],
    final_estimator=LogisticRegression()
)

stacked_model.fit(X_train_scaled, y_train)
y_pred_stack = stacked_model.predict_proba(X_valid_scaled)[:, 1]
print("Stacked Model Log Loss:", log_loss(y_valid, y_pred_stack))
```
✅ **This combines Random Forest, Gradient Boosting, and XGBoost into one model** for better generalization.

---

## **6️⃣ Check for Overfitting**
Look at how well your model performs on the **training set vs. validation set**.

```python
train_pred = best_model.predict_proba(X_train_scaled)[:, 1]
valid_pred = best_model.predict_proba(X_valid_scaled)[:, 1]

train_loss = log_loss(y_train, train_pred)
valid_loss = log_loss(y_valid, valid_pred)

print(f"Training Log Loss: {train_loss}")
print(f"Validation Log Loss: {valid_loss}")
```
✅ If your **training log loss is much lower than validation**, then you might be **overfitting**.

### **Fix Overfitting**
- Increase `train_test_split()` test size (e.g., `test_size=0.3`)
- Use `L1/L2` regularization (`XGBClassifier(reg_alpha=0.1, reg_lambda=1)`)
- Try dropout layers in neural networks

---

## **📌 Summary of Best Strategies**
| 🔥 Method                        | 🎯 Expected Improvement |
|----------------------------------|------------------------|
| **Feature Engineering** (Team stats, Head-to-Head, Streaks) | ✅ Big improvement |
| **Hyperparameter Tuning** (Grid Search for XGBoost) | ✅ Moderate boost |
| **Detailed Game Stats** (Offense, Defense, Efficiency) | ✅ Large improvement |
| **Neural Networks** (Deep learning model) | ✅ Possible improvement |
| **Stacking Model** (Combine RF, XGB, GBM) | ✅ Stronger generalization |
| **Check for Overfitting** (Prevent model bias) | ✅ More reliable model |

---

🏀 **Try these enhancements and let me know which one helps the most!** 🚀

In [1]:
# 🏀 March Madness 2025 - Full Model for Men & Women
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

In [2]:
# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  # Extract numeric part

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

In [3]:
# ✅ Compute Team Statistics (Including FG%, Turnovers, etc.)
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum",
        "WFGM": "mean", "WFGA": "mean",
        "WTO": "mean", "WStl": "mean", "WBlk": "mean"
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames",
                          "WFGM", "WFGA", "Turnovers", "Steals", "Blocks"]

    # Compute Additional Features
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)


In [4]:
# 🏀 Prepare Training Data for Both Men & Women
def prepare_training_data(results, seeds, team_stats):
    # ✅ Merge seeds
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    # ✅ Merge team stats
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # ✅ Handle missing values
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    results["FGMDiff"] = results["FGM%_W"] - results["FGM%_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]

    # ✅ Ensure both datasets have the same columns
    feature_cols = ["WTeamID", "LTeamID", "SeedDiff", "WinRateDiff", "ScoreMarginDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features = loss_features.rename(columns={"WTeamID": "LTeamID", "LTeamID": "WTeamID"})
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

In [5]:
# Process both men's and women's data
mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)

# Combine the datasets
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)


In [6]:
# 🏀 Split and Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [7]:
# 🏀 Train & Evaluate Models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier()
}

best_model = None
best_score = float("inf")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict_proba(X_valid_scaled)[:, 1]
    score = log_loss(y_valid, y_pred)
    print(f"{name} Log Loss: {score}")
    if score < best_score:
        best_score = score
        best_model = model

# 🏀 Cross-Validation Score
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")

Logistic Regression Log Loss: 0.6931030612273088
Random Forest Log Loss: 0.6872654658493319
Gradient Boosting Log Loss: 0.6420874639319117
XGBoost Log Loss: 0.5620806433732296
Cross-Validation Log Loss: 0.6312411756910079


In [8]:
# 🏀 March Madness 2025 - Full Model with Enhancements
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics (Enhanced)
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum",
        "WFGM": "mean", "WFGA": "mean",
        "WTO": "mean", "WStl": "mean", "WBlk": "mean",
        "WLoc": lambda x: (x == "H").sum()  # Home game count
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames",
                          "WFGM", "WFGA", "Turnovers", "Steals", "Blocks", "HomeGames"]

    # Additional Features
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])
    team_stats["HomeWinRate"] = team_stats["HomeGames"] / (team_stats["HomeGames"].sum() + 1)  # Avoid division by zero

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# 🏀 Prepare Training Data (Enhanced Features)
def prepare_training_data(results, seeds, team_stats):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    results["FGMDiff"] = results["FGM%_W"] - results["FGM%_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]
    results["HomeWinRateDiff"] = results["HomeWinRate_W"] - results["HomeWinRate_L"]

    feature_cols = ["WTeamID", "LTeamID", "SeedDiff", "WinRateDiff", "ScoreMarginDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff", "HomeWinRateDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features = loss_features.rename(columns={"WTeamID": "LTeamID", "LTeamID": "WTeamID"})
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Feature Scaling & Transformation
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

poly = PolynomialFeatures(degree=2, interaction_only=True)
X_train_poly = poly.fit_transform(X_train_scaled)
X_valid_poly = poly.transform(X_valid_scaled)

# 🏀 Train & Tune Models
xgb = XGBClassifier()
param_grid = {"learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 300, 500]}
grid_search = GridSearchCV(xgb, param_grid, scoring="neg_log_loss", cv=3)
grid_search.fit(X_train_poly, y_train)

# Best Model
best_model = grid_search.best_estimator_
y_pred = best_model.predict_proba(X_valid_poly)[:, 1]
score = log_loss(y_valid, y_pred)

# 🏀 Cross-Validation Score
cv_scores = cross_val_score(best_model, X_train_poly, y_train, scoring="neg_log_loss", cv=5)

print(f"Optimized XGBoost Log Loss: {score}")
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")


Optimized XGBoost Log Loss: 0.641851036161396
Cross-Validation Log Loss: 0.6576281036707089


In [11]:
# 🏀 March Madness 2025 - Optimized Model
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import shap  # Feature Importance

# 📂 Define Data Path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Elo Ratings
def compute_elo_ratings(results):
    K = 30  # Elo constant
    initial_rating = 1500
    team_ratings = {}

    for index, row in results.iterrows():
        w_team, l_team = row["WTeamID"], row["LTeamID"]

        # Initialize ratings if not available
        if w_team not in team_ratings:
            team_ratings[w_team] = initial_rating
        if l_team not in team_ratings:
            team_ratings[l_team] = initial_rating

        # Compute expected win probabilities
        exp_w = 1 / (1 + 10 ** ((team_ratings[l_team] - team_ratings[w_team]) / 400))
        exp_l = 1 - exp_w

        # Update ratings
        team_ratings[w_team] += K * (1 - exp_w)
        team_ratings[l_team] += K * (0 - exp_l)

    return team_ratings

# Compute Elo Ratings
melo_ratings = compute_elo_ratings(mregular_results)
welo_ratings = compute_elo_ratings(wregular_results)

# ✅ Compute Team Statistics
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean"],
        "LScore": ["mean"],
        "WFGM": ["mean"],
        "WFGA": ["mean"],
        "WTO": ["mean"],
        "WStl": ["mean"],
        "WBlk": ["mean"],
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "AvgLScore", "WFGM", "WFGA", "Turnovers", "Steals", "Blocks"]
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats, elo_ratings):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))
    
    # Merge Elo Ratings
    results["WElo"] = results["WTeamID"].map(elo_ratings)
    results["LElo"] = results["LTeamID"].map(elo_ratings)

    # Fill NaN values
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["AvgWScore_W"] - results["AvgLScore_L"]
    results["FGMDiff"] = results["FGM%_W"] - results["FGM%_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]
    results["EloDiff"] = results["WElo"] - results["LElo"]

    feature_cols = ["SeedDiff", "WinRateDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff", "EloDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats, melo_ratings)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats, welo_ratings)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Feature Scaling
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 🏀 Train Optimized XGBoost
xgb = XGBClassifier(learning_rate=0.005, n_estimators=1000, max_depth=5)
xgb.fit(X_train_scaled, y_train)

y_pred = xgb.predict_proba(X_valid_scaled)[:, 1]
score = log_loss(y_valid, y_pred)

# 🏀 Cross-Validation Score
cv_scores = cross_val_score(xgb, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)

print(f"Optimized XGBoost Log Loss: {score}")
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")


Optimized XGBoost Log Loss: 0.7478472121686537
Cross-Validation Log Loss: 0.7387168897164613


In [12]:
# 🏀 March Madness 2025 - Best Optimized Model
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import shap

# 📂 Define Data Path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Smarter Elo Rating Calculation
def compute_elo_ratings(results):
    K = 30  
    initial_rating = 1400  # Lower base Elo to introduce variability
    team_ratings = {}

    # Give teams that made the tournament in previous years a small Elo boost
    prev_tournament_teams = results.groupby("WTeamID")["Season"].nunique().to_dict()
    
    for team in prev_tournament_teams:
        team_ratings[team] = initial_rating + (prev_tournament_teams[team] * 20)

    for index, row in results.iterrows():
        w_team, l_team = row["WTeamID"], row["LTeamID"]

        if w_team not in team_ratings:
            team_ratings[w_team] = initial_rating
        if l_team not in team_ratings:
            team_ratings[l_team] = initial_rating

        exp_w = 1 / (1 + 10 ** ((team_ratings[l_team] - team_ratings[w_team]) / 400))
        exp_l = 1 - exp_w

        team_ratings[w_team] += K * (1 - exp_w)
        team_ratings[l_team] += K * (0 - exp_l)

    return team_ratings

# Compute Elo Ratings
melo_ratings = compute_elo_ratings(mregular_results)
welo_ratings = compute_elo_ratings(wregular_results)

# ✅ Compute Team Statistics
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean"],
        "LScore": ["mean"],
        "WFGM": ["mean"],
        "WFGA": ["mean"],
        "WTO": ["mean"],
        "WStl": ["mean"],
        "WBlk": ["mean"],
    }).reset_index()

    team_stats.columns = ["Season", "TeamID", "AvgWScore", "AvgLScore", "WFGM", "WFGA", "Turnovers", "Steals", "Blocks"]
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats, elo_ratings):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # Merge Elo Ratings
    results["WElo"] = results["WTeamID"].map(elo_ratings)
    results["LElo"] = results["LTeamID"].map(elo_ratings)

    # Fill NaN values
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering (Reintroducing Select Polynomial Features)
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["AvgWScore_W"] - results["AvgLScore_L"]
    results["FGMDiff"] = results["WFGM_W"] - results["WFGM_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]
    results["EloDiff"] = results["WElo"] - results["LElo"]
    results["SeedDiff^2"] = results["SeedDiff"] ** 2  # Useful polynomial feature

    feature_cols = ["SeedDiff", "SeedDiff^2", "WinRateDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff", "EloDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats, melo_ratings)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats, welo_ratings)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Train Optimized XGBoost (Remove StandardScaler)
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(learning_rate=0.02, n_estimators=500, max_depth=4)
xgb.fit(X_train, y_train)

y_pred = xgb.predict_proba(X_valid)[:, 1]
score = log_loss(y_valid, y_pred)

cv_scores = cross_val_score(xgb, X_train, y_train, scoring="neg_log_loss", cv=5)

print(f"Optimized XGBoost Log Loss: {score}")
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")


Optimized XGBoost Log Loss: 0.7490333015470246
Cross-Validation Log Loss: 0.74698080156519


In [13]:
# 🏀 March Madness 2025 - Optimized Model with Smoothed Elo and Feature Pruning
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import shap

# 📂 Define Data Path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Smoothed Elo Rating Calculation
def compute_smooth_elo(results):
    K = 20  # Lower K-factor for stability
    initial_rating = 1400
    team_ratings = {}

    for team in results["WTeamID"].unique():
        team_ratings[team] = initial_rating

    for index, row in results.iterrows():
        w_team, l_team = row["WTeamID"], row["LTeamID"]

        if w_team not in team_ratings:
            team_ratings[w_team] = initial_rating
        if l_team not in team_ratings:
            team_ratings[l_team] = initial_rating

        # Smooth Elo update with moving average
        exp_w = 1 / (1 + 10 ** ((team_ratings[l_team] - team_ratings[w_team]) / 400))
        team_ratings[w_team] = 0.9 * team_ratings[w_team] + 0.1 * (team_ratings[w_team] + K * (1 - exp_w))
        team_ratings[l_team] = 0.9 * team_ratings[l_team] + 0.1 * (team_ratings[l_team] + K * (0 - exp_w))

    return team_ratings

# Compute Smoothed Elo Ratings
melo_ratings = compute_smooth_elo(mregular_results)
welo_ratings = compute_smooth_elo(wregular_results)

# ✅ Compute Team Statistics
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean"],
        "LScore": ["mean"],
        "WFGM": ["mean"],
        "WFGA": ["mean"],
        "WTO": ["mean"],
        "WStl": ["mean"],
        "WBlk": ["mean"],
    }).reset_index()

    team_stats.columns = ["Season", "TeamID", "AvgWScore", "AvgLScore", "WFGM", "WFGA", "Turnovers", "Steals", "Blocks"]
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats, elo_ratings):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # Merge Smoothed Elo Ratings
    results["WElo"] = results["WTeamID"].map(elo_ratings)
    results["LElo"] = results["LTeamID"].map(elo_ratings)

    # Fill NaN values
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering (Pruning weak features)
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["EloDiff"] = results["WElo"] - results["LElo"]
    results["FGMDiff"] = results["WFGM_W"] - results["WFGM_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]

    # Use SHAP to determine most important features
    feature_cols = ["SeedDiff", "EloDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats, melo_ratings)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats, welo_ratings)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Train XGBoost with New Optimized Settings
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(
    learning_rate=0.05,  # Slightly higher for faster convergence
    n_estimators=600,
    max_depth=5,
    subsample=0.8,  # Introduce some randomness for generalization
    colsample_bytree=0.8
)
xgb.fit(X_train, y_train)

y_pred = xgb.predict_proba(X_valid)[:, 1]
score = log_loss(y_valid, y_pred)

cv_scores = cross_val_score(xgb, X_train, y_train, scoring="neg_log_loss", cv=5)

print(f"Optimized XGBoost Log Loss: {score}")
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")


Optimized XGBoost Log Loss: 0.9374031227697056
Cross-Validation Log Loss: 0.9331038368895245


In [1]:
# 🏀 March Madness 2025 - Optimized Model Fixing Elo and Feature Selection
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import log_loss

# 📂 Define Data Path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])  

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Improved Elo Rating Calculation
def compute_elo(results, K=40):  # Increased K for faster adjustments
    initial_rating = 1500
    team_ratings = {}

    for team in results["WTeamID"].unique():
        team_ratings[team] = initial_rating

    for index, row in results.iterrows():
        w_team, l_team = row["WTeamID"], row["LTeamID"]

        if w_team not in team_ratings:
            team_ratings[w_team] = initial_rating
        if l_team not in team_ratings:
            team_ratings[l_team] = initial_rating

        exp_w = 1 / (1 + 10 ** ((team_ratings[l_team] - team_ratings[w_team]) / 400))
        team_ratings[w_team] += K * (1 - exp_w)
        team_ratings[l_team] += K * (0 - exp_w)

    return team_ratings

# Compute Improved Elo Ratings for Men & Women Separately
melo_ratings = compute_elo(mregular_results, K=40)
welo_ratings = compute_elo(wregular_results, K=35)

# ✅ Compute Team Statistics (Restoring Important Features)
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "WFGM": ["mean"],
        "WFGA": ["mean"],
        "WTO": ["mean"],
        "WStl": ["mean"],
        "WBlk": ["mean"],
    }).reset_index()

    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "WFGM", "WFGA", "Turnovers", "Steals", "Blocks"]
    team_stats["FGM%"] = team_stats["WFGM"] / team_stats["WFGA"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# 🏀 Prepare Training Data (Restore WinRate & ScoreMargin)
def prepare_training_data(results, seeds, team_stats, elo_ratings):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # Merge Elo Ratings
    results["WElo"] = results["WTeamID"].map(elo_ratings)
    results["LElo"] = results["LTeamID"].map(elo_ratings)

    # Fill NaN values
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering (Restored WinRateDiff & ScoreMarginDiff)
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["EloDiff"] = results["WElo"] - results["LElo"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    results["FGMDiff"] = results["WFGM_W"] - results["WFGM_L"]
    results["TurnoverDiff"] = results["Turnovers_W"] - results["Turnovers_L"]
    results["StealDiff"] = results["Steals_W"] - results["Steals_L"]
    results["BlockDiff"] = results["Blocks_W"] - results["Blocks_L"]

    feature_cols = ["SeedDiff", "EloDiff", "WinRateDiff", "ScoreMarginDiff", "FGMDiff", "TurnoverDiff", "StealDiff", "BlockDiff"]
    
    win_features = results[feature_cols].copy()
    win_features["Win"] = 1

    loss_features = results[feature_cols].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats, melo_ratings)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats, welo_ratings)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Train XGBoost with Improved Hyperparameters
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(
    learning_rate=0.03,  # Slightly lower for stability
    n_estimators=400,
    max_depth=4,
    subsample=0.9,  # Higher to allow better learning
    colsample_bytree=0.85
)
xgb.fit(X_train, y_train)

y_pred = xgb.predict_proba(X_valid)[:, 1]
score = log_loss(y_valid, y_pred)

cv_scores = cross_val_score(xgb, X_train, y_train, scoring="neg_log_loss", cv=5)

print(f"Optimized XGBoost Log Loss: {score}")
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")


Optimized XGBoost Log Loss: 0.7743219988164228
Cross-Validation Log Loss: 0.7688867722171476


In [3]:
# 🏀 March Madness 2025 - Best Optimized Model (Fixed)
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler

# 📂 Load Data
data_path = "data/"
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))
mstats = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wstats = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))

# ✅ Extract Numeric Seed Values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ **Better Elo Rating Calculation**
def compute_elo(results, K=45):
    """Compute dynamic Elo ratings that carry over season-to-season."""
    initial_rating = 1500
    team_ratings = {}

    for season in sorted(results["Season"].unique()):
        season_results = results[results["Season"] == season]

        for team in season_results["WTeamID"].unique():
            if team not in team_ratings:
                prev_season = season - 1
                last_rating = team_ratings.get((prev_season, team), initial_rating)
                team_ratings[(season, team)] = last_rating

        for _, row in season_results.iterrows():
            w_team, l_team = row["WTeamID"], row["LTeamID"]
            w_rating = team_ratings.get((season, w_team), initial_rating)
            l_rating = team_ratings.get((season, l_team), initial_rating)

            expected_w = 1 / (1 + 10 ** ((l_rating - w_rating) / 400))
            team_ratings[(season, w_team)] = w_rating + K * (1 - expected_w)
            team_ratings[(season, l_team)] = l_rating + K * (0 - expected_w)

    return team_ratings

# Compute Elo Ratings for Men & Women
melo_ratings = compute_elo(mregular_results, K=45)
welo_ratings = compute_elo(wregular_results, K=40)

# ✅ **Improved Team Statistics**
def compute_team_stats(stats):
    team_stats = stats.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "WFGM": ["mean"],
        "WFGA": ["mean"],
        "WOR": ["mean"],
        "WDR": ["mean"],
        "WAst": ["mean"],
        "WTO": ["mean"],
        "WStl": ["mean"],
        "WBlk": ["mean"],
    }).reset_index()

    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", 
                          "FGM", "FGA", "OffReb", "DefReb", "Assists", "Turnovers", "Steals", "Blocks"]

    team_stats["FGM%"] = team_stats["FGM"] / team_stats["FGA"]
    team_stats["Ast/TO"] = team_stats["Assists"] / (team_stats["Turnovers"] + 1e-5) 
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]

    return team_stats

mteam_stats = compute_team_stats(mstats)
wteam_stats = compute_team_stats(wstats)

# ✅ **Prepare Training Data**
FEATURE_COLS = ["SeedDiff", "EloDiff", "WinRateDiff", "ScoreMarginDiff", "Ast/TO_Diff", "ReboundDiff"]

def prepare_training_data(results, seeds, team_stats, elo_ratings):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    results["WElo"] = results.apply(lambda row: elo_ratings.get((row["Season"], row["WTeamID"]), 1500), axis=1)
    results["LElo"] = results.apply(lambda row: elo_ratings.get((row["Season"], row["LTeamID"]), 1500), axis=1)

    results.fillna(0, inplace=True)

    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["EloDiff"] = results["WElo"] - results["LElo"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    results["Ast/TO_Diff"] = results["Ast/TO_W"] - results["Ast/TO_L"]
    results["ReboundDiff"] = (results["OffReb_W"] + results["DefReb_W"]) - (results["OffReb_L"] + results["DefReb_L"])

    win_features = results[FEATURE_COLS].copy()
    win_features["Win"] = 1
    loss_features = results[FEATURE_COLS].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats, melo_ratings)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats, welo_ratings)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# ✅ **Train XGBoost**
X = full_train_data[FEATURE_COLS]
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(learning_rate=0.02, n_estimators=500, max_depth=6, subsample=0.9, colsample_bytree=0.9)
xgb.fit(X_train, y_train)

print(f"Optimized XGBoost Log Loss: {log_loss(y_valid, xgb.predict_proba(X_valid)[:, 1])}")


Optimized XGBoost Log Loss: 0.853339104433152


In [4]:
# 🏀 March Madness 2025 - Fully Optimized Model
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import RandomizedSearchCV

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics (Advanced)
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum",
        "WFGM": "mean", "WFGA": "mean",
        "WFTM": "mean", "WFTA": "mean",
        "WOR": "mean", "WDR": "mean",
        "WAst": "mean", "WTO": "mean",
        "WStl": "mean", "WBlk": "mean"
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames",
                          "FGM", "FGA", "FTM", "FTA", "OReb", "DReb", "Ast", "TO", "Stl", "Blk"]

    # Compute additional advanced statistics
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])
    team_stats["FG%"] = team_stats["FGM"] / team_stats["FGA"]
    team_stats["FT%"] = team_stats["FTM"] / team_stats["FTA"]
    team_stats["Ast/TO"] = team_stats["Ast"] / (team_stats["TO"] + 1)  # Avoid division by zero
    team_stats["ReboundTotal"] = team_stats["OReb"] + team_stats["DReb"]

    return team_stats

mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats):
    # ✅ Merge seeds
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    # ✅ Merge team stats
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # ✅ Convert non-numeric values to NaN and fill them
    results = results.apply(pd.to_numeric, errors="coerce")
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    results["FG%Diff"] = results["FG%_W"] - results["FG%_L"]
    results["FT%Diff"] = results["FT%_W"] - results["FT%_L"]
    results["Ast/TO_Diff"] = results["Ast/TO_W"] - results["Ast/TO_L"]
    results["ReboundDiff"] = results["ReboundTotal_W"] - results["ReboundTotal_L"]

    # ✅ Create dataset with both win/loss scenarios
    win_features = results[["SeedDiff", "WinRateDiff", "ScoreMarginDiff", "FG%Diff", "FT%Diff", "Ast/TO_Diff", "ReboundDiff"]].copy()
    win_features["Win"] = 1  

    loss_features = results[["SeedDiff", "WinRateDiff", "ScoreMarginDiff", "FG%Diff", "FT%Diff", "Ast/TO_Diff", "ReboundDiff"]].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Train & Optimize XGBoost
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# ✅ Hyperparameter Tuning for XGBoost
xgb = XGBClassifier()
param_grid = {
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "n_estimators": [200, 500, 800],
    "max_depth": [3, 5, 7, 9],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

xgb_search = RandomizedSearchCV(xgb, param_distributions=param_grid, scoring="neg_log_loss", cv=3, n_iter=20, random_state=42, n_jobs=-1)
xgb_search.fit(X_train_scaled, y_train)

# ✅ Evaluate Model
best_xgb = xgb_search.best_estimator_
y_pred = best_xgb.predict_proba(X_valid_scaled)[:, 1]
print(f"Optimized XGBoost Log Loss: {log_loss(y_valid, y_pred)}")

cv_scores = cross_val_score(best_xgb, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")

print(f"Best XGBoost Parameters: {xgb_search.best_params_}")

# ✅ Save Model
import joblib
joblib.dump(best_xgb, "optimized_xgboost_model.pkl")

print("✅ Model training & tuning complete!")


Optimized XGBoost Log Loss: 0.7110838081099462
Cross-Validation Log Loss: 0.7089114510374765
Best XGBoost Parameters: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
✅ Model training & tuning complete!


In [ ]:
# 🏀 March Madness 2025 - Ultimate Model with Feature Engineering, Stacking & Bayesian Optimization
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from skopt import BayesSearchCV  # Bayesian Optimization
from skopt.space import Real, Integer

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonDetailedResults.csv"))  # Use detailed stats
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonDetailedResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Advanced Team Statistics
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum",
        "WFGM": "sum",
        "WFGA": "sum",
        "WFGM3": "sum",
        "WFGA3": "sum",
        "WFTM": "sum",
        "WFTA": "sum",
        "WOR": "sum",
        "WDR": "sum",
        "WAst": "sum",
        "WTO": "sum",
        "WStl": "sum",
        "WBlk": "sum",
    }).reset_index()

    team_stats.columns = [
        "Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore",
        "OTGames", "FGMade", "FGAttempted", "ThreePtMade", "ThreePtAttempted",
        "FTMade", "FTAttempted", "OffRebounds", "DefRebounds",
        "Assists", "Turnovers", "Steals", "Blocks"
    ]

    # 🔹 New Feature Engineering
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])
    team_stats["TurnoverMargin"] = team_stats["Turnovers"] - team_stats["Assists"]
    team_stats["AssistRatio"] = team_stats["Assists"] / (team_stats["FGAttempted"] + team_stats["FTAttempted"])
    team_stats["ThreePointRate"] = team_stats["ThreePtMade"] / team_stats["FGAttempted"]
    team_stats["FreeThrowRate"] = team_stats["FTMade"] / team_stats["FTAttempted"]
    team_stats["ReboundMargin"] = team_stats["OffRebounds"] - team_stats["DefRebounds"]
    team_stats["StealRate"] = team_stats["Steals"] / (team_stats["FGAttempted"] + team_stats["FTAttempted"])
    team_stats["BlockRate"] = team_stats["Blocks"] / (team_stats["FGAttempted"])

    return team_stats

mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# ✅ Prepare Training Data
def prepare_training_data(results, seeds, team_stats):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    results.fillna(0, inplace=True)

    # 🔹 Include newly engineered features
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    results["TurnoverMarginDiff"] = results["TurnoverMargin_W"] - results["TurnoverMargin_L"]
    results["AssistRatioDiff"] = results["AssistRatio_W"] - results["AssistRatio_L"]
    results["ThreePointRateDiff"] = results["ThreePointRate_W"] - results["ThreePointRate_L"]
    results["FreeThrowRateDiff"] = results["FreeThrowRate_W"] - results["FreeThrowRate_L"]
    results["ReboundMarginDiff"] = results["ReboundMargin_W"] - results["ReboundMargin_L"]
    results["StealRateDiff"] = results["StealRate_W"] - results["StealRate_L"]
    results["BlockRateDiff"] = results["BlockRate_W"] - results["BlockRate_L"]

    # ✅ Explicitly define column order
    feature_columns = [
        "WTeamID", "LTeamID", "SeedDiff", "WinRateDiff", "ScoreMarginDiff",
        "TurnoverMarginDiff", "AssistRatioDiff", "ThreePointRateDiff",
        "FreeThrowRateDiff", "ReboundMarginDiff", "StealRateDiff", "BlockRateDiff"
    ]

    win_features = results[feature_columns].copy()
    win_features["Win"] = 1  

    loss_features = results[["LTeamID", "WTeamID"] + feature_columns[2:]].copy()
    loss_features.columns = feature_columns
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)


mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Split and Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 🏀 Stacking Classifier (Model Combination)
base_models = [
    ("rf", RandomForestClassifier(n_estimators=200)),
    ("gb", GradientBoostingClassifier(n_estimators=200)),
    ("xgb", XGBClassifier(n_estimators=200))
]
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())
stack_model.fit(X_train_scaled, y_train)
stack_log_loss = log_loss(y_valid, stack_model.predict_proba(X_valid_scaled)[:, 1])
print(f"Stacked Model Log Loss: {stack_log_loss}")

# 🏀 Bayesian Hyperparameter Optimization for XGBoost
xgb = XGBClassifier()
param_space = {
    "max_depth": Integer(3, 10),
    "learning_rate": Real(0.01, 0.3, "log-uniform"),
    "n_estimators": Integer(100, 1000),
    "subsample": Real(0.7, 1.0),
    "colsample_bytree": Real(0.7, 1.0)
}
opt = BayesSearchCV(xgb, param_space, n_iter=30, scoring="neg_log_loss", cv=3, random_state=42)
opt.fit(X_train_scaled, y_train)

print(f"Optimized XGBoost Log Loss: {-opt.best_score_}")
print(f"Best XGBoost Parameters: {opt.best_params_}")


Stacked Model Log Loss: 0.5539917358234512
Optimized XGBoost Log Loss: 0.6319246437555656
Best XGBoost Parameters: OrderedDict([('colsample_bytree', 0.9443115346100344), ('learning_rate', 0.01), ('max_depth', 9), ('n_estimators', 639), ('subsample', 0.9175415754383407)])


In [12]:
from sklearn.model_selection import RandomizedSearchCV

# 🔹 Define the base XGBoost model
xgb = XGBClassifier()

# 🔹 Define Randomized Search hyperparameter grid
param_grid = {
    'n_estimators': [500, 750, 1000],
    'learning_rate': [0.005, 0.01, 0.02],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# 🔹 Run Randomized Search
search = RandomizedSearchCV(
    xgb, param_distributions=param_grid, n_iter=15,
    scoring='neg_log_loss', cv=3, verbose=2, n_jobs=-1, random_state=42
)
search.fit(X_train_scaled, y_train)

# 🔹 Print best parameters from RandomizedSearchCV
print(f"🔹 Best XGBoost Parameters from RandomizedSearchCV: {search.best_params_}")
print(f"🔹 Optimized XGBoost Log Loss (RandomizedSearchCV): {-search.best_score_}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits
🔹 Best XGBoost Parameters from RandomizedSearchCV: {'subsample': 0.9, 'n_estimators': 750, 'max_depth': 7, 'learning_rate': 0.02, 'colsample_bytree': 1.0}
🔹 Optimized XGBoost Log Loss (RandomizedSearchCV): 0.6368894952147626


In [13]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

# 🔹 Define Bayesian search space
param_space = {
    "max_depth": Integer(3, 10),
    "learning_rate": Real(0.01, 0.3, "log-uniform"),
    "n_estimators": Integer(100, 1000),
    "subsample": Real(0.7, 1.0),
    "colsample_bytree": Real(0.7, 1.0)
}

# 🔹 Run Bayesian Optimization
opt = BayesSearchCV(
    XGBClassifier(), param_space, n_iter=30,
    scoring="neg_log_loss", cv=3, random_state=42, n_jobs=-1
)
opt.fit(X_train_scaled, y_train)

# 🔹 Print best parameters from Bayesian Optimization
print(f"🔹 Best XGBoost Parameters from BayesSearchCV: {opt.best_params_}")
print(f"🔹 Optimized XGBoost Log Loss (BayesSearchCV): {-opt.best_score_}")


🔹 Best XGBoost Parameters from BayesSearchCV: OrderedDict([('colsample_bytree', 0.9443115346100344), ('learning_rate', 0.01), ('max_depth', 9), ('n_estimators', 639), ('subsample', 0.9175415754383407)])
🔹 Optimized XGBoost Log Loss (BayesSearchCV): 0.6319246437555656


In [14]:
# Train final XGBoost model using best parameters from Bayesian Optimization
final_xgb = XGBClassifier(
    colsample_bytree=0.9443,
    learning_rate=0.01,
    max_depth=9,
    n_estimators=639,
    subsample=0.9175
)

# Fit model on full training data
final_xgb.fit(X_train_scaled, y_train)

# Validate the final model
y_pred_final = final_xgb.predict_proba(X_valid_scaled)[:, 1]
final_log_loss = log_loss(y_valid, y_pred_final)

print(f"🏀 Final Optimized XGBoost Log Loss: {final_log_loss}")


🏀 Final Optimized XGBoost Log Loss: 0.5794700527174617


In [16]:
# 📂 Load sample submission file
submission_df = pd.read_csv(os.path.join(data_path, "SampleSubmissionStage2.csv"))

# ✅ Extract Season, WTeamID, and LTeamID
submission_df[["Season", "WTeamID", "LTeamID"]] = submission_df["ID"].str.split("_", expand=True).astype(int)

# ✅ Merge seed information
submission_df = submission_df.merge(mtourney_seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
submission_df = submission_df.merge(mtourney_seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

# ✅ Merge team statistics
submission_df = submission_df.merge(mteam_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
submission_df = submission_df.merge(mteam_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

# ✅ Compute matchup features
submission_df["SeedDiff"] = submission_df["WSeed"] - submission_df["LSeed"]
submission_df["WinRateDiff"] = submission_df["WinRate_W"] - submission_df["WinRate_L"]
submission_df["ScoreMarginDiff"] = submission_df["ScoreMargin_W"] - submission_df["ScoreMargin_L"]
submission_df["TurnoverMarginDiff"] = submission_df["TurnoverMargin_W"] - submission_df["TurnoverMargin_L"]
submission_df["AssistRatioDiff"] = submission_df["AssistRatio_W"] - submission_df["AssistRatio_L"]
submission_df["ThreePointRateDiff"] = submission_df["ThreePointRate_W"] - submission_df["ThreePointRate_L"]
submission_df["FreeThrowRateDiff"] = submission_df["FreeThrowRate_W"] - submission_df["FreeThrowRate_L"]
submission_df["ReboundMarginDiff"] = submission_df["ReboundMargin_W"] - submission_df["ReboundMargin_L"]
submission_df["StealRateDiff"] = submission_df["StealRate_W"] - submission_df["StealRate_L"]
submission_df["BlockRateDiff"] = submission_df["BlockRate_W"] - submission_df["BlockRate_L"]

# ✅ Ensure consistency in feature selection
X_columns = list(X.columns)  # Get feature names used during training
submission_features = submission_df[X_columns]  # Select only matching columns

# ✅ Scale features
submission_features_scaled = scaler.transform(submission_features)

# ✅ Generate predictions
submission_df["Pred"] = final_xgb.predict_proba(submission_features_scaled)[:, 1]

# ✅ Save submission file
submission_df[["ID", "Pred"]].to_csv("submission.csv", index=False)

print("✅ Final submission file generated successfully!")


✅ Final submission file generated successfully!


In [ ]:
# ✅ Print Model Performance
print(f"🔹 Stacked Model Log Loss: {stack_log_loss:.6f}")
print(f"🔹 Optimized XGBoost Log Loss (BayesSearchCV): {-opt.best_score_:.6f}")
print(f"🔹 Best XGBoost Parameters from BayesSearchCV: {opt.best_params_}")
print(f"🔹 Optimized XGBoost Log Loss (RandomizedSearchCV): {abs(search.best_score_):.6f}")
print(f"🔹 Best XGBoost Parameters from RandomizedSearchCV: {search.best_params_}")

# ✅ Cross-Validation Score
cv_scores = cross_val_score(final_xgb, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)
print(f"🔹 Cross-Validation Log Loss: {-np.mean(cv_scores):.6f}")


🔹 Stacked Model Log Loss: 0.553992
🔹 Optimized XGBoost Log Loss (BayesSearchCV): 0.631925
🔹 Best XGBoost Parameters from BayesSearchCV: OrderedDict([('colsample_bytree', 0.9443115346100344), ('learning_rate', 0.01), ('max_depth', 9), ('n_estimators', 639), ('subsample', 0.9175415754383407)])
🔹 Optimized XGBoost Log Loss (RandomizedSearchCV): 0.636889
🔹 Best XGBoost Parameters from RandomizedSearchCV: {'subsample': 0.9, 'n_estimators': 750, 'max_depth': 7, 'learning_rate': 0.02, 'colsample_bytree': 1.0}
🔹 Cross-Validation Log Loss: 0.613616


: 